# Our probe and our editor, on Li et al.'s Othello-GPT

**The question this notebook exists to answer.** On this repo's own world models, a probe reads the
world state well and a probe-derived write to the hidden state does **not** move the generation
(`../../../../research/findings/editability.md`). Two explanations survive: something about *our
world*, or something about *our implementation*. This notebook removes the second one, by taking
**our probe code and our editing code, unmodified**, and running them on a model where the
intervention is published to work — Li et al., *Emergent World Representations* (ICLR 2023,
arXiv:2210.13382), on Othello.

The 2026-08-18 thread `../othello_gpt/` ran the reverse experiment: **their method on our model**.
It found the probing half replicates and the intervention half does not. That leaves our editor
implementation untested, because their write is a different mechanism from ours. This notebook is
the missing direction, and the two are read together.

**What is held fixed.**

| | source | note |
|---|---|---|
| model, tokenizer, board rules, benchmark | **theirs**, unmodified | minGPT `GPT`, `OthelloBoardState`, `intervention_benchmark.pkl` |
| probe module and fitting loop | **ours**, `../othello_gpt/othello_probe.py` | extended in place with a 3-way classification head |
| the write, the descent, the multi-layer schedule | **ours**, byte-identical | `build_edit_spec` · `make_intervention_hook` · `_descend` |
| data and hyperparameter choices | **theirs**, mimicked | synthetic games, `beta = 0.2`, board state as 64 ternary tiles |

The bridge between the two is [`othello_shim.py`](othello_shim.py) — 135 lines supplying the seven
names our editing code calls, and **containing no editing logic of its own**. Cell [2] gates it
against their `GPT.forward`.

**Decision rule.** If our editor reproduces their published intervention result on their model, our
implementation is cleared and the discworld negative is about the world, not the code. If it fails
*here too*, the implementation is implicated and every editability number in the thread needs
re-examination. The published targets are in the table in cell [17].


## Definitions — every term and metric used below

**The world.** An Othello board is 64 tiles, each `white = 0`, `blank = 1`, `black = 2` (their
`OthelloBoardState.get_state`). The model sees a sequence of moves and predicts a distribution over
the next move. The four centre squares are occupied from the start and are never a legal move, so
they carry no token; logits are laid back onto the board with zeros there.

**The intervention.** One tile's colour is flipped. Nothing else about the board changes, and it is
still the same player's turn — but the set of legal next moves changes, by **2.1 squares of 64 on
average**. A successful edit is one where the model's next-move prediction follows the flip.

| term | definition | units | better |
|---|---|---|---|
| **residual point ℓ** | the residual stream **after** ℓ transformer blocks; ℓ = 0 is the embedding, ℓ = 8 the final pre-`ln_f` stream | — | — |
| **applied layer `L_s`** | the earliest residual point written to. The write is then repeated at every point from `L_s` to 8, letting the network recompute in between (their Figure 2C) | — | — |
| **probe error rate** | fraction of the 64 tiles whose class the probe reads wrongly, held-out, × 100 | % | ↓ |
| **frame split** | held out over pooled *(activation, board)* rows — **their** convention (`random_split` in `train_probe_othello.py`). Frames from one game land on both sides | — | — |
| **sequence split** | held out over whole **games** — this repo's convention (`harness/ANALYSIS.md` §2). Not comparable to the frame split; both are always reported, labelled | — | — |
| **Li error vs post-flip** | their §4.2 metric. Take the model's top-*N* predicted moves, *N* = number of legal moves **after** the flip, and count false positives + false negatives against that legal set. Both sets have size *N*, so it equals `2 × (N − overlap)` | errors | ↓ |
| **Li error vs pre-flip** | the identical computation against the legal set **before** the flip. **The guard**: a null intervention is low here and high on the post-flip metric; a successful edit is the reverse; an arm that *destroyed* the model is **high on both**, which their metric alone cannot distinguish | errors | ↑ *(for a successful edit)* |
| **Edit Index (union)** | this repo's `(d_uned − d_edit) / (d_uned + d_edit)` with `d_·` = RMSE of the predicted distribution against a ground-truth world, scored on the squares where the two worlds differ. The reference world is **uniform over legal moves**, which is exact here rather than approximate: their generator draws moves uniformly from the legal set. Support = the **union** of the two legal sets, because the two references renormalise (1/\|L₀\| vs 1/\|L₁\|) and so differ on shared legal squares too, in 69.9% of cases | −1…+1 | ↑ |
| **Edit Index (symdiff)** | the same, scored only on squares whose **legality** changed. A narrower question; reported alongside and **never quoted as the same quantity** | −1…+1 | ↑ |
| **legal mass** | total predicted probability on the post-flip legal set | 0…1 | ↑ |
| **hit target** | fraction of cases where the probe, after the write, reads the *requested* class at the flipped tile. This is their own success criterion (`num_error == 0`) and it is what selects the step size — **never** the outcome metric | 0…1 | ↑ |
| **hold rest** | the same for the other 63 tiles, which the write is supposed to leave alone | 0…1 | ↑ |
| **`beta`** | weight on the hold-the-rest term in the edit objective; their `reg_strg`. `beta = 0.2` throughout, their value | — | — |
| **`alpha`** | descent step size, **relative** to each residual point's activation scale (`probe.act_scale`), so one value means the same size of write at every depth | — | — |

**Calibration for the Edit Index.** Measured on all 1001 cases, 2026-08-20: a *perfect* predictor of
the unedited world scores exactly **−1** on both supports. The real unedited model scores
**−0.829 (union)** and **−0.943 (symdiff)** — it is not quite perfect, sitting 0.0016 RMSE per square
from its own reference against a 0.0193 separation between the two worlds, a 12× margin. Those two
numbers, not −1, are the floor an editor has to beat.


## Runs and arms used here

**No world models were trained.** Rows copied from [`OTHELLO_TRANSFER_RUNS.md`](OTHELLO_TRANSFER_RUNS.md).

| code | descriptive label | source | architecture | provenance |
|---|---|---|---|---|
| `gpt_synthetic` | **Othello-GPT · trained on synthetic games** | Li et al. (ICLR 2023) | 8 blocks, 8 heads, `d_model` 512, `block_size` 59, vocab 61, 25.3 M params | the authors' Google Drive link is **dead**; recovered from the HuggingFace mirror `sbentley/othello-world-ckpts`, then **verified** functionally identical to the authors' own TransformerLens conversion `NeelNanda/Othello-GPT-Transformer-Lens` (max probability difference 2.1e-6) and reproducing the paper's legal-move property (99.98% of mass on the legal set) |

**Probe arms.** One probe per *(target, family, split, residual point)* — never shared across points,
matching both their setup and `../othello_gpt/`.

| arm | family | why |
|---|---|---|
| **MLP 512 hidden** | one hidden layer, width 512 | **ours** — the exact width `../othello_gpt/` used on this repo's transformer, so the two threads' probes are the same object |
| **MLP 128 hidden** | one hidden layer, width 128 | **theirs** — the width behind their intervention checkpoints (`state_tl128`) |
| **linear** | no hidden layer | their §3.1 probe. Trained by the same SGD loop, because a 3-way classifier has no `lstsq` closed form — so unlike the regression path in `../othello_gpt/`, the linear arm here carries **no optimiser advantage** |

| target | labels | why |
|---|---|---|
| **absolute colour** | white / blank / black | **theirs**, and the priority. Their reported error rates are for this target |
| **mine / theirs** | blank / mine / theirs, relative to the player to move | a one-line relabel. Nanda's finding is that the board is linearly decodable in *these* coordinates and their linear probe missed it only because absolute colour alternates sign every move |

**Intervention arms.** All use the probes fit under **their** frame split, so the probe matches what
they intervened through; cell [16] repeats the operating point with sequence-split probes as a
robustness check.

| arm | `n_steps` | why |
|---|---|---|
| **ours, 100 steps** | 100 | the default `../othello_gpt/` ran on this repo's transformer |
| **theirs, 1000 steps** | 1000 | their literal Appendix G setting |

`L_s` is swept over all nine residual points. `alpha` is selected by **read-out convergence** — the
smallest value at which the probe reads the requested board — never by the outcome metric. That rule
exists because on 2026-08-19 the operating point chosen for `../othello_gpt/` turned out to
understate that method roughly 3× (`../../../../research/GOTCHAS.md`).


In [ ]:
# [1] Setup — imports, paths, provenance, and every knob in one place.
import json, os, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

THREAD = Path.cwd()
REPO = THREAD.parents[3]
OTHELLO_ROOT = Path("/home/sevan/research/PIM/othello_world")
FIGDIR = REPO / "runs" / "othello_transfer" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)
for p in (str(THREAD), str(THREAD.parent / "othello_gpt"), str(OTHELLO_ROOT), str(REPO)):
    if p not in sys.path:
        sys.path.insert(0, p)

import othello_data as od
import othello_probe as op
import transfer_pipeline as tp
import board_grid as bg
from pim.figures.theme import PALETTE, style_ax

# ── configuration ────────────────────────────────────────────────────────────
# SMOKE=1 runs the identical pipeline at a size that finishes in ~2 min. It exists to shake
# out plumbing before committing an hour of compute, and its numbers are NOT reportable —
# the probe is deliberately undertrained. Every reported figure comes from a SMOKE=0 run.
SMOKE = os.environ.get("PIM_OT_SMOKE", "0") == "1"

SEED = 0
N_GAMES = 1_500 if SMOKE else 20_000   # ~1.18 M (activation, board) rows; their scale was ~130k games
EPOCHS, BATCH, LR = (8 if SMOKE else 200), 4096, 1e-3   # ../othello_gpt/'s probe settings, unchanged
HOLDOUT = 0.2
BETA = 0.2                  # their `reg_strg`
ALPHAS = (0.005, 0.05) if SMOKE else (0.002, 0.005, 0.01, 0.02, 0.05, 0.1)
N_STEPS_OURS, N_STEPS_LI = (20, 50) if SMOKE else (100, 1000)
START_LAYERS = (3, 4) if SMOKE else tuple(range(tp.N_POINTS))
HEADLINE_FAMILY = "MLP 512 hidden"   # ours; "MLP 128 hidden" is theirs and is run alongside
PANEL_SEED, N_PANEL = 0, 4

# published numbers we are trying to hit (Li et al. 2023, natural benchmark)
LI = {"null_baseline": 2.68, "best_intervention": 0.12, "best_Ls": 4,
      "nonlinear_probe_error": 1.7, "linear_probe_error": 20.4}

torch.manual_seed(SEED)
np.random.seed(SEED)
t_notebook = time.time()
RESULTS = {}

print(f"mode        : {'SMOKE — numbers are not reportable' if SMOKE else 'full run'}")
print(f"device      : {tp.DEVICE}  ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'})")
print(f"checkpoint  : {tp.CKPT}")
print(f"othello repo: {OTHELLO_ROOT}")
print(f"figures     : {FIGDIR}")


In [ ]:
# [2] Correctness gates. Everything downstream is meaningless if any of these fails, so they
#     run first and assert rather than print. The bridge is the risky part: it must reproduce
#     their forward pass exactly, or "our editor on their model" is not what is being measured.
from mingpt.model import GPT, GPTConfig, GPTforProbing
from mingpt.dataset import CharDataset

shim = tp.load_model()
_cfg = GPTConfig(61, 59, n_layer=8, n_head=8, n_embd=512)
_g = torch.randint(0, 61, (4, 23), device=tp.DEVICE, generator=torch.Generator(tp.DEVICE).manual_seed(0))

# (a) the block stack is bit-identical to their own forward pass
with torch.no_grad():
    _ref = shim.gpt(_g)[0]
    _h, _ = shim._run(shim.embed(_g), None)
    _mine = shim.decoder(shim.norm_out(_h))
assert torch.equal(_mine[:, -1], _ref[:, -1]), "shim forward differs from GPT.forward"

# (b) every residual point matches what THEIR GPTforProbing hands a probe
_sd = torch.load(tp.CKPT, map_location="cpu", weights_only=True)
_rs = shim.residual_stack(_g)
for _ell in range(tp.N_POINTS):
    _p = GPTforProbing(_cfg, probe_layer=_ell)
    _p.load_state_dict(_sd)
    with torch.no_grad():
        assert torch.equal(_rs[_ell], _p.to(tp.DEVICE).eval()(_g)), f"residual point {_ell} differs"
    del _p

# (c) a no-op hook leaves the pass unchanged; a real write reaches the logits
_noop = lambda layer, x: x
with torch.no_grad():
    _h2, _ = shim._run(shim.embed(_g), None, edit=_noop)
    assert torch.equal(_h2, _h), "a no-op hook changed the forward pass"
    _zero = lambda layer, x: x if layer != 4 else torch.cat([x[:, :-1], x[:, -1:] * 0], 1)
    _h3, _ = shim._run(shim.embed(_g), None, edit=_zero)
    assert (shim.decoder(shim.norm_out(_h3[:, -1])) - _ref[:, -1]).abs().max() > 1.0, "the hook never reached the stream"

# (d) the benchmark is what we think it is
bench = od.load_benchmark()
assert bench.n_cases == 1001 and len(bench.tokens) == 26
assert all(len(L) > 0 for L in bench.legal_post), "a case leaves the side to move with no legal move"
assert all(set(a) != set(b) for a, b in zip(bench.legal_pre, bench.legal_post)), "a flip changes nothing"

# (e) the model reproduces the paper's core property before we ask anything harder of it
_p_un = tp.unsteered(shim, bench)
_mass_pre = float(np.mean([_p_un[i, L].sum() for i, L in enumerate(bench.legal_pre)]))
assert _mass_pre > 0.99, f"legal-move mass {_mass_pre:.4f} — this is not a working Othello-GPT"

del _ref, _h, _h2, _h3, _mine, _rs, _sd
torch.cuda.empty_cache()
print("all correctness gates passed")
print(f"  shim forward and all {tp.N_POINTS} residual points are bit-identical to their code")
print(f"  benchmark: {bench.n_cases} cases in {len(bench.tokens)} equal-length buckets")
print(f"  model's probability mass on the pre-flip legal set: {_mass_pre:.4f}")


In [ ]:
# [3] Probe training data — synthetic games from THEIR generator, labelled with THEIR board rules.
#     Their own script hardcodes `data_root="data/othello_championship"` even for this model, but
#     that data is behind a dead Google Drive link; synthetic is also the distribution this
#     checkpoint was trained on. ~15 s.
t0 = time.time()
games = od.synthetic_games(N_GAMES, seed=SEED)
od.assert_vocab_matches(CharDataset(games))
data = od.tokens_and_labels(games)
n_rows = int(data.mask.sum())

print(f"{len(games):,} games in {time.time() - t0:.1f}s   "
      f"lengths {data.lengths.min()}–{data.lengths.max()} (median {int(np.median(data.lengths))})")
print(f"{n_rows:,} (activation, board) rows   tokens {data.tokens.shape}   labels {data.labels.shape}")
print(f"vocabulary matches CharDataset, block_size 59")
for name, y in (("absolute colour", data.labels), ("mine / theirs", data.mine)):
    frac = np.bincount(y[data.mask].reshape(-1), minlength=3) / y[data.mask].size
    print(f"  {name:16s} class balance: {frac[0]:.3f} / {frac[1]:.3f} / {frac[2]:.3f}"
          f"   -> majority-class error rate {100 * (1 - frac.max()):.2f}%")
RESULTS["n_games"], RESULTS["n_rows"] = len(games), n_rows


In [ ]:
# [4] Fit the probe grid: 2 targets x 3 families x 2 splits x 9 residual points = 108 probes,
#     each fit independently on its own residual point. This is `../othello_gpt/`'s probe and its
#     fitting loop, with only a 3-way classification head added. ~40 min.
t0 = time.time()
grid = tp.fit_probe_grid(
    shim, data,
    targets=("state", "mine"),
    families=("MLP 512 hidden", "MLP 128 hidden", "linear"),
    splits=("frame", "sequence"),
    holdout=HOLDOUT, epochs=EPOCHS, batch=BATCH, lr=LR, seed=SEED,
    log=lambda s: print(s, flush=True),
)
print(f"\n{len(grid.stats)} probes fit in {(time.time() - t0) / 60:.1f} min")
RESULTS["probe_stats"] = grid.stats

TARGET_LABEL = {"state": "absolute colour", "mine": "mine / theirs"}
SPLIT_LABEL = {"frame": "frame split (theirs)", "sequence": "sequence split (ours)"}


def pstat(target, family, split, point):
    return next(s for s in grid.stats if s["target"] == target and s["family"] == family
                and s["split"] == split and s["point"] == point)


def pcurve(target, family, split):
    return [pstat(target, family, split, i)["error_rate"] for i in range(tp.N_POINTS)]


In [ ]:
# [5] Fig 1 — probe error rate across residual points. Their Tables 1 and 2 as a figure.
#     Absolute error rates, not gains; each panel carries the published reference lines it should
#     be read against, and the majority-class rate as the do-nothing floor. Linear is dashed in
#     every panel, MLP solid, so the family is readable without consulting the legend.
FAM_STYLE = {"MLP 512 hidden": dict(color=PALETTE[0], ls="-", marker="o"),
             "MLP 128 hidden": dict(color=PALETTE[3], ls="-", marker="^"),
             "linear": dict(color=PALETTE[1], ls="--", marker="s")}
SPLIT_STYLE = {"frame": dict(color=PALETTE[0], ls="-", marker="o"),
               "sequence": dict(color=PALETTE[4], ls="--", marker="D")}
x = np.arange(tp.N_POINTS)
majority = pstat("state", HEADLINE_FAMILY, "frame", 0)["majority_class_error_rate"]

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6), facecolor="white")

for fam in FAM_STYLE:
    axes[0].plot(x, pcurve("state", fam, "frame"), label=fam, lw=2, ms=6, **FAM_STYLE[fam])
axes[0].axhline(LI["nonlinear_probe_error"], color="0.35", ls=":", lw=1.6,
                label=f"Li et al. nonlinear, best layer ({LI['nonlinear_probe_error']}%)")
axes[0].axhline(LI["linear_probe_error"], color="0.6", ls=":", lw=1.6,
                label=f"Li et al. linear, best layer ({LI['linear_probe_error']}%)")
axes[0].set_title("(a) probe family — absolute colour, frame split (theirs)")

for split in SPLIT_STYLE:
    axes[1].plot(x, pcurve("state", HEADLINE_FAMILY, split), label=SPLIT_LABEL[split],
                 lw=2, ms=6, **SPLIT_STYLE[split])
axes[1].axhline(LI["nonlinear_probe_error"], color="0.35", ls=":", lw=1.6,
                label=f"Li et al. nonlinear, best layer ({LI['nonlinear_probe_error']}%)")
axes[1].set_title(f"(b) held-out convention — absolute colour, {HEADLINE_FAMILY}")

for tgt, col in (("state", PALETTE[0]), ("mine", PALETTE[2])):
    axes[2].plot(x, pcurve(tgt, HEADLINE_FAMILY, "sequence"), color=col, ls="-", marker="o",
                 label=f"{HEADLINE_FAMILY} — {TARGET_LABEL[tgt]}", lw=2, ms=6)
    axes[2].plot(x, pcurve(tgt, "linear", "sequence"), color=col, ls="--", marker="s",
                 label=f"linear — {TARGET_LABEL[tgt]}", lw=2, ms=6)
axes[2].set_title("(c) coordinate frame — sequence split (ours)")

for ax in axes:
    ax.axhline(majority, color="0.75", ls="-.", lw=1.4, label=f"majority class ({majority:.1f}%)")
    ax.set_xlabel("residual point (0 = embedding, 8 = final pre-LayerNorm stream)")
    ax.set_ylabel("probe error rate, held out (%)  — lower is better")
    ax.set_xticks(x)
    ax.set_ylim(0, max(60, majority + 5))
    ax.legend(fontsize=7.5, handlelength=2.6, loc="upper right")
    style_ax(ax)
fig.suptitle("Fig 1 — how well our probe reads Li et al.'s board state, by depth", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGDIR / "fig1_probe_quality.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# [6] Table 1 — probe error rate, every family x split x residual point, plus the MLP >= linear
#     tripwire. A strictly more expressive probe scoring WORSE than a linear one is a training
#     failure, never a fact about the representation; it has caught two real bugs in this repo
#     (2026-08-11, 2026-08-19), so it is checked here rather than assumed. Its MAGNITUDE is
#     reported too: the rule is a tripwire for a real gap, and two probes both sitting at the
#     ceiling of an easy target can cross it by a rounding error without anything being wrong.
rows = ["| target | held-out convention | probe family | " + " | ".join(f"ℓ{i}" for i in x) + " | best |",
        "|---|---|---|" + "---|" * (tp.N_POINTS + 1)]
for tgt in ("state", "mine"):
    for split in ("frame", "sequence"):
        for fam in FAM_STYLE:
            c = pcurve(tgt, fam, split)
            rows.append(f"| {TARGET_LABEL[tgt]} | {SPLIT_LABEL[split]} | {fam} | "
                        + " | ".join(f"{v:.2f}" for v in c) + f" | **{min(c):.2f}** |")
display(Markdown("**Table 1 — probe error rate (%), held out. Lower is better.**\n\n" + "\n".join(rows)))

viol = [(t, s, p, pstat(t, HEADLINE_FAMILY, s, p)["error_rate"], pstat(t, "linear", s, p)["error_rate"])
        for t in ("state", "mine") for s in ("frame", "sequence") for p in range(tp.N_POINTS)
        if pstat(t, HEADLINE_FAMILY, s, p)["error_rate"] > pstat(t, "linear", s, p)["error_rate"]]
print(f"MLP >= linear tripwire: {len(viol)} violation(s) of {2 * 2 * tp.N_POINTS} comparisons")
for t, s, p, m, l in viol:
    print(f"  ! {TARGET_LABEL[t]} / {SPLIT_LABEL[s]} / point {p}: MLP {m:.2f}% vs linear {l:.2f}%  "
          f"(gap {m - l:+.2f} pp)")
if viol:
    worst = max(m - l for *_, m, l in viol)
    tgts = {t for t, *_ in viol}
    print(f"  largest gap {worst:.2f} pp; violations confined to: {', '.join(sorted(TARGET_LABEL[t] for t in tgts))}")
    print("  Reading: a violation matters when the MLP is leaving decodable structure on the table.")
    print("  Where both probes are already near the floor of an easy target, a sub-0.2 pp crossing is")
    print("  the two of them at the ceiling, not an undertrained MLP — check which case this is against")
    print("  the absolute-colour rows, where the ordering should hold by a wide margin.")
RESULTS["mlp_ge_linear_violation_detail"] = [
    {"target": t, "split": s, "point": p, "mlp": m, "linear": l} for t, s, p, m, l in viol]

best = {}
for tgt in ("state", "mine"):
    for split in ("frame", "sequence"):
        for fam in FAM_STYLE:
            c = pcurve(tgt, fam, split)
            best[(tgt, fam, split)] = (int(np.argmin(c)), float(min(c)))
print()
for k in (("state", HEADLINE_FAMILY, "frame"), ("state", "MLP 128 hidden", "frame"),
          ("state", "linear", "frame"), ("state", HEADLINE_FAMILY, "sequence"),
          ("mine", "linear", "sequence"), ("mine", HEADLINE_FAMILY, "sequence")):
    p, v = best[k]
    print(f"best {TARGET_LABEL[k[0]]:16s} {k[1]:14s} {SPLIT_LABEL[k[2]]:22s}: {v:6.2f}% at point {p}")
print(f"\nLi et al. report {LI['nonlinear_probe_error']}% (nonlinear) and {LI['linear_probe_error']}% "
      f"(linear) at their best layer, on a FRAME split.")
RESULTS["probe_best"] = {f"{a}|{b}|{c}": v for (a, b, c), v in best.items()}
RESULTS["mlp_ge_linear_violations"] = len(viol)


In [ ]:
# [7] The unsteered arm — no intervention, computed through the identical code path. This is the
#     null-intervention baseline of their §4.2 and the floor every editor is read against.
probes_frame = {f: {p: grid.probes[("state", f, "frame", p)] for p in range(tp.N_POINTS)}
                for f in ("MLP 512 hidden", "MLP 128 hidden")}
probes_seq = {p: grid.probes[("state", HEADLINE_FAMILY, "sequence", p)] for p in range(tp.N_POINTS)}

probs_unsteered = tp.unsteered(shim, bench)
card_unsteered = od.scorecard(probs_unsteered, bench)
RESULTS["unsteered"] = card_unsteered

print("null intervention (no write), all 1001 cases")
for k in ("li_error_vs_post", "li_error_vs_pre", "edit_index_union", "edit_index_symdiff", "legal_mass"):
    print(f"  {k:20s} {card_unsteered[k]:+.4f}")
print(f"\n  Li et al. report a null-intervention baseline of {LI['null_baseline']} on the natural "
      f"benchmark and {2.59} on the unnatural one.")
print(f"  Ours is {card_unsteered['li_error_vs_post']:.3f} -> this is the NATURAL subset.")


In [ ]:
# [8] The step-size sweep, at their best applied layer.
#
#     ⚠ NO write-side criterion selects a step size in this setting, and that is a result rather
#     than a nuisance. Two were tried:
#       * `hit_target` — does the probe read the requested board? Their own success rule
#         (`num_error == 0`). It reads 1.000 at EVERY alpha across a 50x range.
#       * the edit objective's own value — the continuous quantity behind that argmax. It reaches
#         >= 99% of its best reduction at every alpha too.
#     Meanwhile the outcome varies by ~100x over the same range. The probe constraint is therefore
#     satisfiable everywhere, and what decides whether the dynamics honour the write is its
#     MAGNITUDE, not its satisfaction of the probe. Same shape as the 2026-08-18 finding on this
#     repo's own transformer — "the optimiser decides which probe-satisfying write you land on" —
#     reproduced here on their model.
#
#     Rather than manufacture a rule that would pick a point for no reason, cell [10] runs the
#     applied-layer sweep at BOTH ends of the range and reports both. The replication claim is read
#     off this sweep, exactly as Li et al. read theirs off their own best configuration. ~10 min.
t0 = time.time()
sweep = {}
for n_steps in (N_STEPS_OURS, N_STEPS_LI):
    for a in ALPHAS:
        pr, rec = tp.run_arm(shim, bench, probes_frame[HEADLINE_FAMILY], LI["best_Ls"],
                             alpha=a, n_steps=n_steps, beta=BETA)
        card = od.scorecard(pr, bench)
        pts = [ell for ell in rec if ell >= LI["best_Ls"]]
        sweep[(n_steps, a)] = {
            "hit_target_min": float(np.min([rec[e]["hit_target_after"] for e in pts])),
            "hold_rest": float(np.mean([rec[e]["hold_rest_after"] for e in pts])),
            "edit_loss_before": float(np.mean([rec[e]["edit_loss_before"] for e in pts])),
            "edit_loss_after": float(np.mean([rec[e]["edit_loss_after"] for e in pts])),
            "write_ratio": float(np.mean([rec[e]["delta_norm"] / rec[e]["x_norm"] for e in pts])),
            **{k: v for k, v in card.items() if not k.endswith("per_case")},
        }
        s = sweep[(n_steps, a)]
        print(f"  n_steps {n_steps:4d}  alpha {a:<6.3f}  hit_target {s['hit_target_min']:.3f}  "
              f"edit loss {s['edit_loss_before']:.4f} -> {s['edit_loss_after']:.4f}  "
              f"|dx|/|x| {s['write_ratio']:.2f}  ->  Li error {s['li_error_vs_post']:6.3f} "
              f"(vs pre {s['li_error_vs_pre']:6.3f})   Edit Index {s['edit_index_union']:+.3f}", flush=True)
print(f"\nsweep in {(time.time() - t0) / 60:.1f} min")
RESULTS["sweep"] = {f"{n}|{a}": v for (n, a), v in sweep.items()}

# ── document the degeneracy, mechanically, instead of selecting through it ────
CONVERGED, SATURATED_SPREAD = 0.99, 0.01
LOSS_FRAC, DEGENERATE = {}, {}
for n_steps in (N_STEPS_OURS, N_STEPS_LI):
    row = {a: sweep[(n_steps, a)] for a in ALPHAS}
    before = row[ALPHAS[0]]["edit_loss_before"]
    span = before - min(row[a]["edit_loss_after"] for a in ALPHAS)
    frac = {a: ((before - row[a]["edit_loss_after"]) / span if span > 0 else 1.0) for a in ALPHAS}
    LOSS_FRAC[n_steps] = frac
    hits = [row[a]["hit_target_min"] for a in ALPHAS]
    outcomes = [row[a]["li_error_vs_post"] for a in ALPHAS]
    DEGENERATE[n_steps] = {
        "hit_target_saturated": bool(max(hits) - min(hits) < SATURATED_SPREAD),
        "loss_saturated": bool(min(frac.values()) >= CONVERGED),
        "outcome_span_factor": float(max(outcomes) / max(min(outcomes), 1e-9)),
    }
    d = DEGENERATE[n_steps]
    print(f"n_steps {n_steps}: hit_target spans {min(hits):.3f}-{max(hits):.3f}"
          f"{'  [SATURATED]' if d['hit_target_saturated'] else ''}   "
          f"edit-loss reduction spans {min(frac.values()):.3f}-{max(frac.values()):.3f}"
          f"{'  [SATURATED]' if d['loss_saturated'] else ''}")
    print(f"           outcome (Li error) spans {min(outcomes):.3f}-{max(outcomes):.3f} "
          f"= a factor of {d['outcome_span_factor']:.0f} over the same range")

ALPHA_ARMS = (min(ALPHAS), 0.02)
ALPHA_ARM_LABEL = {ALPHA_ARMS[0]: "smallest probe-satisfying write",
                   ALPHA_ARMS[1]: "best of the swept range"}
print(f"\napplied-layer arms run at alpha = {ALPHA_ARMS}, both reported:")
for a in ALPHA_ARMS:
    print(f"  {a:<6} — {ALPHA_ARM_LABEL[a]}")
RESULTS["alpha_degeneracy"] = {str(k): v for k, v in DEGENERATE.items()}
RESULTS["alpha_loss_reduction_fraction"] = {f"{n}|{a}": f for n, d in LOSS_FRAC.items() for a, f in d.items()}
RESULTS["alpha_arms"] = list(ALPHA_ARMS)

In [ ]:
# [9] Fig 2 — the step-size sweep. Panel (a) shows why no step size can be selected from the
#     write side here: both write-side criteria are flat near 1.0 across the whole range, while
#     panels (b) and (c) show the outcome moving by two orders of magnitude over the same range.
fig, axes = plt.subplots(1, 3, figsize=(17.5, 4.8), facecolor="white")
NS_STYLE = {N_STEPS_OURS: dict(color=PALETTE[0], ls="-", marker="o"),
            N_STEPS_LI: dict(color=PALETTE[4], ls="--", marker="D")}
NS_LABEL = {N_STEPS_OURS: f"{N_STEPS_OURS} steps (ours)", N_STEPS_LI: f"{N_STEPS_LI} steps (theirs)"}

for ns in NS_STYLE:
    axes[0].plot(ALPHAS, [LOSS_FRAC[ns][a] for a in ALPHAS], lw=2, ms=6,
                 label=f"edit-loss reduction — {NS_LABEL[ns]}", **NS_STYLE[ns])
    st = dict(NS_STYLE[ns]); st["color"] = PALETTE[1] if ns == N_STEPS_OURS else PALETTE[2]
    axes[0].plot(ALPHAS, [sweep[(ns, a)]["hit_target_min"] for a in ALPHAS], lw=1.6, ms=5,
                 alpha=0.9, label=f"probe reads requested board — {NS_LABEL[ns]}", **st)
axes[0].axhline(CONVERGED, color="0.4", ls=":", lw=1.6, label=f"convergence threshold ({CONVERGED})")
for a in ALPHA_ARMS:
    axes[0].axvline(a, color="0.55", ls="-.", lw=1.4, alpha=0.8,
                    label=f"arm at alpha={a} — {ALPHA_ARM_LABEL[a]}")
axes[0].set_ylabel("fraction of the criterion achieved (0–1)")
axes[0].set_title("(a) write-side convergence — saturated, so it cannot select")
axes[0].set_ylim(0, 1.06)

for ns in NS_STYLE:
    axes[1].plot(ALPHAS, [sweep[(ns, a)]["li_error_vs_post"] for a in ALPHAS],
                 label=f"vs post-flip world — {NS_LABEL[ns]}", lw=2, ms=6, **NS_STYLE[ns])
    st = dict(NS_STYLE[ns]); st["color"] = PALETTE[1] if ns == N_STEPS_OURS else PALETTE[2]
    axes[1].plot(ALPHAS, [sweep[(ns, a)]["li_error_vs_pre"] for a in ALPHAS],
                 label=f"vs pre-flip world (guard) — {NS_LABEL[ns]}", lw=1.6, ms=5, alpha=0.9, **st)
axes[1].axhline(card_unsteered["li_error_vs_post"], color="0.4", ls=":", lw=1.6,
                label=f"no intervention, vs post-flip ({card_unsteered['li_error_vs_post']:.2f})")
axes[1].axhline(LI["best_intervention"], color="0.7", ls="--", lw=1.6,
                label=f"Li et al. best ({LI['best_intervention']})")
axes[1].set_yscale("log")
axes[1].set_ylabel("Li error vs legal set (log)")
axes[1].set_title("(b) outcome, absolute — both worlds on one axis")

for ns in NS_STYLE:
    axes[2].plot(ALPHAS, [sweep[(ns, a)]["edit_index_union"] for a in ALPHAS],
                 label=NS_LABEL[ns], lw=2, ms=6, **NS_STYLE[ns])
axes[2].axhline(card_unsteered["edit_index_union"], color="0.4", ls=":", lw=1.6,
                label=f"no intervention ({card_unsteered['edit_index_union']:+.3f})")
axes[2].axhline(1.0, color="0.7", ls="--", lw=1.6, label="a perfect edit (+1)")
axes[2].axhline(0.0, color="0.88", lw=1.0)
axes[2].set_ylabel("Edit Index (union support) — at the intervention, step 0")
axes[2].set_title("(c) outcome, Edit Index — comparable to this repo's own models")
axes[2].set_ylim(-1.05, 1.05)

for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("alpha — descent step, relative to each residual point's activation scale")
    ax.legend(fontsize=6.5, handlelength=2.4, loc='best')
    style_ax(ax)
fig.suptitle(f"Fig 2 — step-size sweep at applied layer L_s = {LI['best_Ls']}, all 1001 cases", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.94), w_pad=2.2)
fig.savefig(FIGDIR / "fig2_step_size_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [10] Every arm: applied layer L_s over all nine residual points, at both declared step sizes,
#      for both probe widths and both `n_steps`. This is their Figure 3, widened by the alpha axis
#      that cell [8] showed cannot be collapsed. ~35 min.
t0 = time.time()
ARMS = {}


def arm_label(fam, n_steps, alpha, Ls):
    who = "ours" if n_steps == N_STEPS_OURS else "theirs"
    return f"{fam} · {n_steps} steps ({who}) · alpha={alpha} · L_s={Ls}"


for fam in ("MLP 512 hidden", "MLP 128 hidden"):
    for n_steps in (N_STEPS_OURS, N_STEPS_LI):
        for alpha in ALPHA_ARMS:
            for Ls in START_LAYERS:
                pr, rec = tp.run_arm(shim, bench, probes_frame[fam], Ls,
                                     alpha=alpha, n_steps=n_steps, beta=BETA)
                card = od.scorecard(pr, bench)
                pts = [e for e in rec if e >= Ls]
                card["hit_target"] = float(np.mean([rec[e]["hit_target_after"] for e in pts]))
                card["hold_rest"] = float(np.mean([rec[e]["hold_rest_after"] for e in pts]))
                card["write_ratio"] = float(np.mean([rec[e]["delta_norm"] / rec[e]["x_norm"] for e in pts]))
                ARMS[(fam, n_steps, alpha, Ls)] = {"probs": pr, "card": card}
                print(f"  {arm_label(fam, n_steps, alpha, Ls):<64s} Li error {card['li_error_vs_post']:6.3f} "
                      f"(vs pre {card['li_error_vs_pre']:6.3f})  Edit Index {card['edit_index_union']:+.3f}",
                      flush=True)
print(f"\n{len(ARMS)} arms in {(time.time() - t0) / 60:.1f} min")
RESULTS["arms"] = {f"{f}|{n}|{a}|{L}": {k: v for k, v in d["card"].items() if not k.endswith("per_case")}
                   for (f, n, a, L), d in ARMS.items()}
RESULTS["arms_edit_index_per_case"] = {
    f"{f}|{n}|{a}|{L}": d["card"]["edit_index_union_per_case"] for (f, n, a, L), d in ARMS.items()}

In [ ]:
# [11] Fig 3 — the headline. Their §4.2 metric and this repo's Edit Index, by applied layer,
#      plotted ABSOLUTE (never as a gain), with the null baseline and their published best on the
#      same axis. One column per step size; both columns carry the identical four arms in the
#      identical order, so the panels can be scanned horizontally.
ARM_STYLE = {("MLP 512 hidden", N_STEPS_OURS): dict(color=PALETTE[0], ls="-", marker="o"),
             ("MLP 512 hidden", N_STEPS_LI): dict(color=PALETTE[4], ls="--", marker="D"),
             ("MLP 128 hidden", N_STEPS_OURS): dict(color=PALETTE[3], ls="-", marker="^"),
             ("MLP 128 hidden", N_STEPS_LI): dict(color=PALETTE[2], ls="--", marker="v")}


def arm_curve(fam, ns, alpha, key):
    return [ARMS[(fam, ns, alpha, L)]["card"][key] for L in START_LAYERS]


fig, axes = plt.subplots(2, len(ALPHA_ARMS), figsize=(7.2 * len(ALPHA_ARMS), 9.4), facecolor="white")
for j, alpha in enumerate(ALPHA_ARMS):
    for (fam, ns), st in ARM_STYLE.items():
        who = "ours" if ns == N_STEPS_OURS else "theirs"
        lab = f"{fam} · {ns} steps ({who})"
        axes[0, j].plot(START_LAYERS, arm_curve(fam, ns, alpha, "li_error_vs_post"),
                        label=lab, lw=2, ms=6, **st)
        axes[1, j].plot(START_LAYERS, arm_curve(fam, ns, alpha, "edit_index_union"),
                        label=lab, lw=2, ms=6, **st)
    axes[0, j].axhline(card_unsteered["li_error_vs_post"], color="0.35", ls=":", lw=1.8,
                       label=f"no intervention ({card_unsteered['li_error_vs_post']:.2f})")
    axes[0, j].axhline(LI["best_intervention"], color="0.65", ls="--", lw=1.8,
                       label=f"Li et al. best, L_s={LI['best_Ls']} ({LI['best_intervention']})")
    axes[0, j].set_yscale("log")
    axes[0, j].set_ylabel("Li error vs the post-flip legal set (log)  — lower is better")
    axes[0, j].set_title(f"(a{j + 1}) their metric — alpha = {alpha} ({ALPHA_ARM_LABEL[alpha]})")
    axes[1, j].axhline(card_unsteered["edit_index_union"], color="0.35", ls=":", lw=1.8,
                       label=f"no intervention ({card_unsteered['edit_index_union']:+.3f})")
    axes[1, j].axhline(1.0, color="0.65", ls="--", lw=1.8, label="a perfect edit (+1)")
    axes[1, j].axhline(0.0, color="0.88", lw=1.0)
    axes[1, j].set_ylim(-1.05, 1.05)
    axes[1, j].set_ylabel("Edit Index (union support) — at the intervention, step 0")
    axes[1, j].set_title(f"(b{j + 1}) this repo's metric — alpha = {alpha}")
for ax in axes.flat:
    ax.set_xlabel("applied layer $L_s$ — earliest residual point written to")
    ax.set_xticks(START_LAYERS)
    ax.legend(fontsize=7.5, handlelength=2.6)
    style_ax(ax)
fig.suptitle("Fig 3 — our editor on their model, by applied layer (all 1001 benchmark cases)", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(FIGDIR / "fig3_headline_by_applied_layer.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [12] Fig 4 — the guard. Their metric alone cannot tell "the edit worked" from "the model was
#      destroyed", because both move it. Scoring against BOTH worlds separates them: a null
#      intervention sits bottom-right, a successful edit top-left, a destroyed model top-right.
fig, axes = plt.subplots(1, len(ALPHA_ARMS), figsize=(6.9 * len(ALPHA_ARMS), 6.4),
                         facecolor="white", sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for j, alpha in enumerate(ALPHA_ARMS):
    ax = axes[j]
    for (fam, ns), st in ARM_STYLE.items():
        who = "ours" if ns == N_STEPS_OURS else "theirs"
        xs = arm_curve(fam, ns, alpha, "li_error_vs_post")
        ys = arm_curve(fam, ns, alpha, "li_error_vs_pre")
        ax.plot(xs, ys, lw=1.4, alpha=0.6, **{k: v for k, v in st.items() if k != "marker"})
        ax.scatter(xs, ys, s=[26 + 10 * L for L in START_LAYERS], color=st["color"],
                   marker=st["marker"], label=f"{fam} · {ns} steps ({who})", zorder=3)
    ax.scatter([card_unsteered["li_error_vs_post"]], [card_unsteered["li_error_vs_pre"]],
               s=170, marker="*", color="0.2", zorder=4, label="no intervention")
    ax.axvline(LI["best_intervention"], color="0.65", ls="--", lw=1.6,
               label=f"Li et al. best vs post-flip ({LI['best_intervention']})")
    ax.axvline(card_unsteered["li_error_vs_post"], color="0.35", ls=":", lw=1.4,
               label=f"no intervention, vs post-flip ({card_unsteered['li_error_vs_post']:.2f})")
    ax.set_xscale("symlog", linthresh=0.01)
    ax.annotate("a successful edit\nlands here", xy=(0.04, 0.95), xycoords="axes fraction",
                fontsize=9, color="0.25", ha="left", va="top")
    ax.annotate("degraded,\nnot steered", xy=(0.97, 0.95), xycoords="axes fraction",
                fontsize=9, color="0.25", ha="right", va="top")
    ax.set_xlabel("Li error vs the POST-flip legal set (symlog)  — lower means the edit landed")
    ax.set_title(f"alpha = {alpha} — {ALPHA_ARM_LABEL[alpha]}", fontsize=11)
    if j == 0:
        ax.set_ylabel("Li error vs the PRE-flip legal set  — higher means it left the old world")
        ax.legend(fontsize=7.5, loc="center left", handlelength=2.2)
    style_ax(ax)
fig.suptitle("Fig 4 — the guard: an edit must move TOWARD the new world and AWAY from the old.\n"
             "Marker size grows with the applied layer $L_s$.", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(FIGDIR / "fig4_guard_both_worlds.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [13] Table 2 — the full scorecard for every arm, and the best arm identified mechanically.
hdr = ("| probe family | steps | alpha | $L_s$ | Li error vs post ↓ | Li error vs pre ↑ | "
       "Edit Index union ↑ | Edit Index symdiff ↑ | legal mass ↑ | hit target ↑ | hold rest ↑ | ‖Δx‖/‖x‖ |")
rows = [hdr, "|---|---|---|---|---|---|---|---|---|---|---|---|",
        f"| *no intervention* | — | — | — | {card_unsteered['li_error_vs_post']:.3f} | "
        f"{card_unsteered['li_error_vs_pre']:.3f} | {card_unsteered['edit_index_union']:+.3f} | "
        f"{card_unsteered['edit_index_symdiff']:+.3f} | {card_unsteered['legal_mass']:.3f} | — | — | — |"]
for fam in ("MLP 512 hidden", "MLP 128 hidden"):
    for ns in (N_STEPS_OURS, N_STEPS_LI):
        for alpha in ALPHA_ARMS:
            for L in START_LAYERS:
                c = ARMS[(fam, ns, alpha, L)]["card"]
                rows.append(f"| {fam} | {ns} | {alpha} | {L} | {c['li_error_vs_post']:.3f} | "
                            f"{c['li_error_vs_pre']:.3f} | {c['edit_index_union']:+.3f} | "
                            f"{c['edit_index_symdiff']:+.3f} | {c['legal_mass']:.3f} | "
                            f"{c['hit_target']:.3f} | {c['hold_rest']:.3f} | {c['write_ratio']:.2f} |")
display(Markdown("**Table 2 — every intervention arm, all 1001 cases.**\n\n" + "\n".join(rows)))

BEST = min(ARMS, key=lambda k: ARMS[k]["card"]["li_error_vs_post"])
BEST_EI = max(ARMS, key=lambda k: ARMS[k]["card"]["edit_index_union"])
bc, bec = ARMS[BEST]["card"], ARMS[BEST_EI]["card"]
SWEEP_BEST = min(sweep, key=lambda k: sweep[k]["li_error_vs_post"])
sc = sweep[SWEEP_BEST]
print(f"best arm by THEIR metric : {arm_label(*BEST)}")
print(f"    Li error vs post {bc['li_error_vs_post']:.3f}  (no intervention {card_unsteered['li_error_vs_post']:.3f}, "
      f"Li et al. {LI['best_intervention']})")
print(f"    Li error vs pre  {bc['li_error_vs_pre']:.3f}  (no intervention {card_unsteered['li_error_vs_pre']:.3f})")
print(f"    Edit Index union {bc['edit_index_union']:+.3f}  (no intervention {card_unsteered['edit_index_union']:+.3f})")
print(f"    legal mass       {bc['legal_mass']:.3f}  (no intervention {card_unsteered['legal_mass']:.3f})")
print(f"best arm by EDIT INDEX   : {arm_label(*BEST_EI)}   Edit Index {bec['edit_index_union']:+.3f}")
print(f"best point in the STEP-SIZE SWEEP (L_s={LI['best_Ls']}): n_steps {SWEEP_BEST[0]}, alpha {SWEEP_BEST[1]}"
      f"  ->  Li error {sc['li_error_vs_post']:.3f}, Edit Index {sc['edit_index_union']:+.3f}")
RESULTS["best_arm_li"] = {"key": list(map(str, BEST)), **{k: v for k, v in bc.items() if not k.endswith("per_case")}}
RESULTS["best_arm_edit_index"] = {"key": list(map(str, BEST_EI)), **{k: v for k, v in bec.items() if not k.endswith("per_case")}}
RESULTS["best_sweep_point"] = {"key": list(map(str, SWEEP_BEST)), **sc}

In [ ]:
# [14] Fig 5 — the qualitative panel. Samples are drawn at RANDOM (the selection rule is in the
#      title); an extreme-case panel is a biased draw whenever effect size correlates with the
#      selection variable, which cost this thread a misread on 2026-08-18.
import pickle
from data.othello import OthelloBoardState

rng = np.random.default_rng(PANEL_SEED)
panel_cases = sorted(rng.choice(bench.n_cases, N_PANEL, replace=False).tolist())
with open(OTHELLO_ROOT / "intervention_benchmark.pkl", "rb") as f:
    _ds = pickle.load(f)

boards_pre, boards_post = {}, {}
for ci in panel_cases:
    b = OthelloBoardState()
    b.update(_ds[ci]["history"], prt=False)
    boards_pre[ci] = b.state.copy()
    sq = bench.pos_int[ci]
    b.state[sq // 8, sq % 8] = bench.new_class[ci] - 1
    boards_post[ci] = b.state.copy()

arms_for_panel = {"no intervention": (probs_unsteered, f"mean Li error {card_unsteered['li_error_vs_post']:.3f}")}
for alpha in ALPHA_ARMS:
    key = (HEADLINE_FAMILY, N_STEPS_OURS, alpha, LI["best_Ls"])
    arms_for_panel[f"alpha={alpha} · $L_s$={LI['best_Ls']}"] = (
        ARMS[key]["probs"], f"mean Li error {ARMS[key]['card']['li_error_vs_post']:.3f}")
if BEST not in [(HEADLINE_FAMILY, N_STEPS_OURS, a, LI["best_Ls"]) for a in ALPHA_ARMS]:
    arms_for_panel[f"best arm · $L_s$={BEST[3]}"] = (ARMS[BEST]["probs"], f"mean Li error {bc['li_error_vs_post']:.3f}")

fig = bg.board_panel(
    bench, boards_pre, boards_post, arms_for_panel, panel_cases, fig_no=5, seed=PANEL_SEED,
    subtitle=(f"All arms use {HEADLINE_FAMILY} probes. Column metrics are POPULATION MEANS over all "
              f"1001 cases, not this row's value. Read against: Li error "
              f"{card_unsteered['li_error_vs_post']:.2f} with no intervention,\n"
              f"{bc['li_error_vs_post']:.3f} for the best arm, {LI['best_intervention']} reported by Li et al."))
fig.savefig(FIGDIR / "fig5_board_panel.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [15] Robustness — does the held-out convention used to FIT the probe change the intervention?
#      Everything above intervenes through frame-split probes, because that is how theirs were fit.
#      This repeats the best arm with sequence-split probes. If the two agree, the split convention
#      is a reporting choice about probe accuracy and not a confound in the edit.
pr_seq, rec_seq = tp.run_arm(shim, bench, probes_seq, BEST[3],
                             alpha=BEST[2], n_steps=BEST[1], beta=BETA)
card_seq = od.scorecard(pr_seq, bench)
_pts = [e for e in rec_seq if e >= BEST[3]]
card_seq["hit_target"] = float(np.mean([rec_seq[e]["hit_target_after"] for e in _pts]))
RESULTS["best_arm_sequence_split_probe"] = {k: v for k, v in card_seq.items() if not k.endswith("per_case")}

rows = ["| probe fit under | Li error vs post ↓ | Li error vs pre ↑ | Edit Index union ↑ | hit target ↑ |",
        "|---|---|---|---|---|",
        f"| frame split (theirs) | {bc['li_error_vs_post']:.3f} | {bc['li_error_vs_pre']:.3f} | "
        f"{bc['edit_index_union']:+.3f} | {bc['hit_target']:.3f} |",
        f"| sequence split (ours) | {card_seq['li_error_vs_post']:.3f} | {card_seq['li_error_vs_pre']:.3f} | "
        f"{card_seq['edit_index_union']:+.3f} | {card_seq['hit_target']:.3f} |"]
display(Markdown(f"**Table 3 — the same arm ({arm_label(*BEST)}), probes fit under each convention.**\n\n"
                 + "\n".join(rows)))
print(f"difference in Li error vs post: {abs(bc['li_error_vs_post'] - card_seq['li_error_vs_post']):.3f}")

In [ ]:
# [16] Table 4 — the comparison this notebook exists for. Three columns: what Li et al. report on
#      their model with their code, what we get on their model with OUR code, and what the same
#      code gets on this repo's own transformer (imported from ../othello_gpt/, cited not recomputed).
OGPT = {  # ../othello_gpt/othello_gpt_probing.ipynb, 2026-08-18, corrected 2026-08-19
    "probe_note": "position R² 0.798 linear → 0.934 MLP (regression, not an error rate)",
    "unsteered_edit_index": -0.684,
    "best_edit_index": -0.194,
    "source": "`../othello_gpt/` on `runs/transformers/W16`, dataset `4_fixed_refl_inview`",
}
BEST_ANY = min([bc["li_error_vs_post"], sc["li_error_vs_post"]])
BEST_ANY_EI = max([bec["edit_index_union"]] + [sweep[k]["edit_index_union"] for k in sweep])
pb = RESULTS["probe_best"]
rows = [
    "| quantity | Li et al., their code, their model | **ours, our code, their model** | our code, our model |",
    "|---|---|---|---|",
    f"| nonlinear probe, best layer | {LI['nonlinear_probe_error']}% error (frame split) | "
    f"**{pb['state|' + HEADLINE_FAMILY + '|frame'][1]:.2f}% error** (frame split) · "
    f"{pb['state|' + HEADLINE_FAMILY + '|sequence'][1]:.2f}% (sequence split) | {OGPT['probe_note']} |",
    f"| linear probe, best layer | {LI['linear_probe_error']}% error | "
    f"**{pb['state|linear|frame'][1]:.2f}% error** (absolute colour) · "
    f"{pb['mine|linear|frame'][1]:.2f}% (mine/theirs) | — |",
    f"| null intervention | {LI['null_baseline']} | **{card_unsteered['li_error_vs_post']:.3f}** | "
    f"Edit Index {OGPT['unsteered_edit_index']:+.3f} |",
    f"| best intervention | {LI['best_intervention']} (at $L_s$={LI['best_Ls']}) | "
    f"**{BEST_ANY:.3f}** | Edit Index {OGPT['best_edit_index']:+.3f} |",
    f"| Edit Index, no intervention | not reported | **{card_unsteered['edit_index_union']:+.3f}** | "
    f"{OGPT['unsteered_edit_index']:+.3f} |",
    f"| Edit Index, best arm | not reported | **{BEST_ANY_EI:+.3f}** | {OGPT['best_edit_index']:+.3f} |",
]
display(Markdown("**Table 4 — replication scorecard.** The our-model column is cited from "
                 f"{OGPT['source']}, not recomputed here.\n\n" + "\n".join(rows)))

theirs_x = LI["null_baseline"] / max(LI["best_intervention"], 1e-9)
ours_x = card_unsteered["li_error_vs_post"] / max(BEST_ANY, 1e-9)
reproduced = BEST_ANY <= LI["best_intervention"] * 2
print("\nDECISION RULE — does our editor reproduce their intervention on their model?")
print(f"  their null baseline      {LI['null_baseline']}      ours {card_unsteered['li_error_vs_post']:.3f}")
print(f"  their best intervention  {LI['best_intervention']}      ours {BEST_ANY:.3f}")
print(f"  error reduction:  theirs {theirs_x:.0f}x   ours {ours_x:.0f}x")
print(f"  -> {'REPRODUCED' if reproduced else 'NOT REPRODUCED'} "
      f"(rule: our best within 2x of their published {LI['best_intervention']})")
print("\n  Guard: the best arm keeps legal mass at "
      f"{bc['legal_mass']:.3f} (no intervention {card_unsteered['legal_mass']:.3f}) and moves the "
      f"pre-flip error from {card_unsteered['li_error_vs_pre']:.3f} to {bc['li_error_vs_pre']:.3f} —"
      " it left the old world rather than degrading.")
RESULTS["reproduced"] = bool(reproduced)
RESULTS["best_any_li_error"] = float(BEST_ANY)
RESULTS["best_any_edit_index"] = float(BEST_ANY_EI)
RESULTS["li_reference"] = LI
RESULTS["othello_gpt_reference"] = OGPT

In [ ]:
# [17] Serialise everything, curves included. A `{k: v for ... if not isinstance(v, list)}` filter on
#      the way to JSON has silently dropped the per-case curves a required plot depends on more than
#      once in this repo (`harness/ANALYSIS.md` §1) — so the per-case arrays are written out too.
out = REPO / "runs" / "othello_transfer" / "results.json"
RESULTS["config"] = {
    "seed": SEED, "n_games": N_GAMES, "epochs": EPOCHS, "batch": BATCH, "lr": LR,
    "holdout": HOLDOUT, "beta": BETA, "alphas": list(ALPHAS),
    "n_steps_ours": N_STEPS_OURS, "n_steps_li": N_STEPS_LI,
    "start_layers": list(START_LAYERS), "headline_family": HEADLINE_FAMILY,
    "panel_seed": PANEL_SEED, "n_panel": N_PANEL, "convergence_threshold": CONVERGED,
    "checkpoint": str(tp.CKPT), "device": tp.DEVICE,
}
RESULTS["runtime_minutes"] = (time.time() - t_notebook) / 60
out.write_text(json.dumps(RESULTS, indent=1, default=float))
print(f"wrote {out}  ({out.stat().st_size / 1e6:.1f} MB)")
print(f"figures in {FIGDIR}:")
for f in sorted(FIGDIR.glob("*.png")):
    print(f"  {f.name}  ({f.stat().st_size / 1e3:.0f} KB)")
print(f"\ntotal notebook runtime: {RESULTS['runtime_minutes']:.1f} min")


## Summary

### Current results (updated 2026-08-20)

**Our probe and our editor, unmodified, reproduce Li et al.'s intervention on Li et al.'s model —
and comfortably exceed the published number.**

| | Li et al. | **ours, our code, their model** |
|---|---|---|
| null intervention | 2.68 | **2.723** |
| best intervention | 0.12 (at `L_s` = 4) | **0.016** |
| error reduction | 22× | **170×** |
| nonlinear probe, best layer | 1.7% error | **0.57%** |
| linear probe, best layer | 20.4% error | 23.90% (absolute colour) · **0.72%** (mine/theirs) |

**This closes the implementation question.** Every editability number in this thread comes from
`../othello_gpt/othello_probe.py` — its probe fitting, its edit objective, its activation descent,
its multi-layer schedule. Until now nothing could test that code independently, because it *was* the
instrument. Run unmodified on the model the published result was published on, through a bridge
gated bit-identical to their own forward pass, it works. **The discworld editability negative is not
a bug in our editor.**

**On the Edit Index — the axis that puts both worlds on one scale** (Fig 3, panel b2):

| | unsteered | best arm | gain | crosses zero |
|---|---|---|---|---|
| Othello — their model, our code | **−0.829** | **+0.656** | +1.49 | **yes** |
| discworld `W16` — our model, our code | −0.684 | −0.194 | +0.49 | **no** |

81% of the available headroom against 29%, and the sign is the substance: on Othello the output
stops being the unedited world and becomes the edited one. On the symmetric-difference support the
sweep reaches **+0.868** against a −0.943 floor.

**The guard confirms it steered rather than degraded.** The best arm moves legal mass
0.858 → **0.998** and pre-flip error 0.002 → **2.214**. Li et al. report no such guard; at α = 0.1
arms visibly degrade (Fig 4, up-and-right), which their metric alone cannot see.

**Three secondary results.**

1. **Nanda's linear finding reproduces exactly** (Fig 1c). Linear probe at its best layer: **23.90%**
   error in absolute colour, **0.72%** in mine/theirs. Li's "linear probes fail" is a
   coordinate-frame artifact — the board was linearly decodable all along, and their nonlinear probe
   was reading a representation that did not need to be nonlinear.
2. **Their frame-level split does not inflate their number** (Fig 1b). 0.57% (frame) vs 0.66%
   (sequence) — about 0.1 pp, against the +0.34 R² inflation the same convention change causes on
   discworld (`../../../../research/GOTCHAS.md`, 2026-08-14). Board state changes enough per move
   that little leaks. Cell [15] further shows the probe's split convention does not move the
   intervention outcome.
3. **The probe constraint does not pin down the write on their model either** (Fig 2a). `hit_target`
   — the probe reads the requested board, their own success criterion — is **1.000 at every alpha
   across a 50× range**, and the edit objective reaches ≥99% of its best reduction throughout, while
   the outcome moves by a factor of **83**. No write-side criterion can select a step size here.
   What decides whether the dynamics honour the write is its **magnitude**, not its satisfaction of
   the probe. That makes the 2026-08-18 discworld observation — *the optimiser decides which
   probe-satisfying write you land on* — a property of probe-derived writes generally.

**Sharp transition at `L_s` = 5** (Fig 3). Writing only at the last three or four residual points
fails, the Edit Index collapsing from ~+0.6 to −0.5, matching the structural prediction that a write
at residual point ℓ changes block inputs only for layers > ℓ.

**The two metrics disagree about the best step size.** Li error is minimised at α = 0.05; the Edit
Index at α = 0.02. "Closest to the new world" is not "cleanly *is* the new world" — which is the
distinction the Edit Index exists to draw.

**Caveat kept in view.** The MLP ≥ linear tripwire fires 8 times, all on mine/theirs, all with both
probes under 3.2% and a largest gap of 0.19 pp. Read as two probes at the ceiling of an easy target
rather than an undertrained MLP — on absolute colour the ordering holds by 23 pp — but not dismissed.

### What this notebook can and cannot conclude

**Can.** Whether *this repo's probe-fitting code and its activation-space editor* reproduce a
published intervention result on the model that result was published on. That is a statement about
the implementation and nothing else.

**Cannot.** It does not discriminate between the two explanations left open by
`../othello_gpt/` (2026-08-18) for why the same code fails on this repo's own world model:

* **the world** — Othello's board is discrete and is consumed *directly* by the legal-move
  computation, while discworld's object positions are continuous and reach the output only through
  a renderer. Consistent with 2026-08-05 locating `readable ≠ grabbable` in the world.
* **the read-out** — their probe predicts a quantity the next-token computation demonstrably
  consumes; ours predicts one merely correlated with it.

Both survive a positive result here. What it removes is the third possibility, live since
2026-07-08, that the editor code was simply wrong.

Everything here is **step 0 only**, matching their benchmark. Persistence — where discworld's edits
actually die (+0.146 → +0.010 by step 14) — is untested here and needs its own design.

### Deviations from Li et al., all deliberate

1. **Synthetic games, not championship.** Their probe script hardcodes the championship set even for
   this checkpoint, but that data is behind a dead Google Drive link, and synthetic is this model's
   own training distribution. (~93% of the championship set is reconstructible from the public WTHOR
   archive if ever needed — 136,055 games, parsed and validated 2026-08-20.)
2. **`model.eval()`.** Their harvest runs with dropout live at p = 0.1, since
   `train_probe_othello.py` never disables it.
3. **Input standardisation inside the probe.** Ours takes a raw activation and standardises with a
   floored per-dim scale; theirs does not. It is what makes activation-space descent well
   conditioned across residual points differing by an order of magnitude in scale, and it is why
   their `lr = 1e-3` does not transfer — hence the sweep in cell [8].
4. **20k games, not ~130k.** A 6× smaller probe training set.
5. **Both held-out conventions reported**, never merged.